In [ ]:
# Read the pdf

file_path = "../Data/odisha_judgement_files/display_judgement20pdf.pdf"
# file_path = "/Users/m2sm/Desktop/projects/Agentic-AI/Judgement_Bot/Data/odisha_judgement_files/display_judgement20pdf.pdf"

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path)

documents = loader.load()
documents

invalid pdf header: b'\r\n\xef\xbb\xbf'
incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'GPL Ghostscript 10.01.1; modified using eSign™ 5.5.13.2 ©2000-2020', 'creator': 'PDF24 Creator', 'creationdate': '2023-07-06T11:02:00+05:30', 'moddate': '2023-07-06T11:06:52+05:30', 'source': '/Users/m2sm/Desktop/projects/Agentic-AI/Judgement_Bot/Data/odisha_judgement_files/display_judgement20pdf.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1'}, page_content='CRLMC No.1462 of 2023                                                              Page 1 of 25 \n \n  \n \n           IN THE HIGH COURT OF ORISSA AT CUTTACK \n \nCRLMC NO.1462 OF 2023 \n \n(Application under Section 482 of the Code of Criminal  \nProcedure for fresh investigation or re-investigation by any \nindependent agency) \n \n            \n            Bandhna Toppo    \n                                                   …     Petitioner \n              \n     -versus-  \n \nState of Orissa  \nand others                      …     Opposite Parties \n \n                                 

# Traditional chunking and retrieval



In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=[
            "\n\n",           # Split by major paragraphs first
            "\n",             # Then split by line breaks
            ". ",             # Then split by sentences
            ", ",             # Then split by clauses
            " "               # Last resort: split by words
        ]
)

In [9]:

raw_chunks = text_splitter.split_documents(documents)

In [11]:
len(raw_chunks)

48

In [18]:
type(raw_chunks[0])

langchain_core.documents.base.Document

In [24]:
raw_chunks[0].metadata


{'producer': 'GPL Ghostscript 10.01.1; modified using eSign™ 5.5.13.2 ©2000-2020',
 'creator': 'PDF24 Creator',
 'creationdate': '2023-07-06T11:02:00+05:30',
 'moddate': '2023-07-06T11:06:52+05:30',
 'source': '/Users/m2sm/Desktop/projects/Agentic-AI/Judgement_Bot/Data/odisha_judgement_files/display_judgement20pdf.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1'}

In [23]:
raw_chunks[0].metadata['source'].split('/')[-1].split('.')[0]

'display_judgement20pdf'

In [26]:
# raw_chunks

from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("BAAI/bge-large-en-v1.5")






python(91276) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3949.53it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
import chromadb
collection_name: str = "J_files_recursive"
batch_size: int = 250
host: str = "manishs-mac-mini.tailc96719.ts.net"
port: int = 8030

client = chromadb.HttpClient(host=host, port=port)

collection = client.get_or_create_collection(name=collection_name)

chunk_no =0
for chunk in raw_chunks:
    chunk_no += 1
    id = chunk.metadata['source'].split('/')[-1].replace('.pdf','') + "_chunk_" + str(chunk_no)
    source = chunk.metadata['source']
    total_pages = chunk.metadata['total_pages']
    page_number = chunk.metadata['page']
    page_label = chunk.metadata['page_label']
    embedding = encoder.encode(chunk.page_content)
    collection.add(
        ids=id,
        embeddings=embedding,
        documents=chunk.page_content,
        metadatas={'source':source, 'total_pages':total_pages, 'page_number':page_number, 'page_label':page_label}
    )



[Document(metadata={'producer': 'GPL Ghostscript 10.01.1; modified using eSign™ 5.5.13.2 ©2000-2020', 'creator': 'PDF24 Creator', 'creationdate': '2023-07-06T11:02:00+05:30', 'moddate': '2023-07-06T11:06:52+05:30', 'source': '/Users/m2sm/Desktop/projects/Agentic-AI/Judgement_Bot/Data/odisha_judgement_files/display_judgement20pdf.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1'}, page_content='CRLMC No.1462 of 2023                                                              Page 1 of 25 \n \n  \n \n           IN THE HIGH COURT OF ORISSA AT CUTTACK \n \nCRLMC NO.1462 OF 2023 \n \n(Application under Section 482 of the Code of Criminal  \nProcedure for fresh investigation or re-investigation by any \nindependent agency) \n \n            \n            Bandhna Toppo    \n                                                   …     Petitioner \n              \n     -versus-  \n \nState of Orissa  \nand others                      …     Opposite Parties \n \n                                 

In [ ]:
# smart chunking and retrieval


